# FLEO — Rebuttal runs (Comments 3.4 + 2.9)

Kaggle, RAF-DB, YOLOv8n. Runs:
- **3.4 control**: baseline 40 ep, then +10 ep @ lr0=5e-4 (does plain extra training explain the recovery?)
- **2.9 comparison**: YOLO11-FLEO and YOLO26-FLEO.

Settings: `Add Input` the RAF-DB dataset, Accelerator = GPU, Internet = On. Then Run All.
~2 h total on a P100.

## 0. Setup — clone, install, prepare RAF-DB

In [ ]:
import os, subprocess
os.chdir('/kaggle/working')
if not os.path.isdir('FLEO'):
    subprocess.run(['git','clone','https://github.com/olfa-askri/FLEO.git'])
os.chdir('/kaggle/working/FLEO')
subprocess.run(['git','pull'])
subprocess.run(['pip','install','-q','ultralytics','onnx','onnxruntime'])

# find the RAF-DB mount automatically, then prepare it
raf = None
for r, ds, fs in os.walk('/kaggle/input'):
    if 'raf-db-dataset' in os.path.basename(r).lower() or r.lower().endswith('raf-db-dataset'):
        raf = r; break
if raf is None:
    # fallback: any path containing 'raf'
    for r, ds, fs in os.walk('/kaggle/input'):
        if 'raf' in r.lower():
            raf = r; break
print('RAF-DB source:', raf)
subprocess.run(['python','-m','data.prepare_rafdb','--src', raf, '--out','datasets/rafdb'])
print('OK — datasets/rafdb:', os.listdir('datasets/rafdb'))

## 1. Comment 3.4a — baseline reference (40 epochs)

In [ ]:
import os, subprocess, sys
os.chdir('/kaggle/working/FLEO')
subprocess.run([sys.executable,'-m','scripts.train','--data','datasets/rafdb/data.yaml',
  '--variant','baseline','--cfg','yolov8n.yaml','--pretrained','yolov8n.pt',
  '--epochs','40','--imgsz','128','--batch','16','--device','0','--seeds','0',
  '--workers','2','--project','runs/ctrl_base40'])

## 2. Comment 3.4b — control: +10 epochs @ lr0=5e-4

In [ ]:
import os, glob, subprocess, sys
os.chdir('/kaggle/working/FLEO')
base = sorted(glob.glob('runs/ctrl_base40/**/weights/best.pt', recursive=True))[-1]
print('continue from:', base)
subprocess.run([sys.executable,'-m','scripts.train','--data','datasets/rafdb/data.yaml',
  '--variant','baseline','--cfg','yolov8n.yaml','--pretrained', base,
  '--epochs','10','--lr0','0.0005','--imgsz','128','--batch','16','--device','0',
  '--seeds','0','--workers','2','--project','runs/ctrl_base_plus10'])

## 3. Comment 2.9 — YOLO11-FLEO and YOLO26-FLEO

In [ ]:
import os, subprocess, sys
os.chdir('/kaggle/working/FLEO')
for cfg, pt, name in [('yolo11n.yaml','yolo11n.pt','y11'), ('yolo26n.yaml','yolo26n.pt','y26')]:
    print(f'\n===== {name} =====', flush=True)
    r = subprocess.run([sys.executable,'-m','scripts.train','--data','datasets/rafdb/data.yaml',
      '--variant','fleo','--cfg',cfg,'--pretrained',pt,'--d','8','--fuzzy-alpha','0.1',
      '--epochs','40','--imgsz','128','--batch','16','--device','0','--seeds','0',
      '--workers','2','--project',f'runs/{name}'])
    print(f'{name} exit code {r.returncode} (if !=0 e.g. cfg not found, skip it)')

## 4. Collect all results

In [ ]:
import os, glob, pandas as pd
os.chdir('/kaggle/working/FLEO')
def best(proj):
    c = sorted(glob.glob(f'{proj}/**/results.csv', recursive=True))
    if not c: return ('-','-')
    df = pd.read_csv(c[0]); df.columns = [x.strip() for x in df.columns]
    b = df.loc[df['metrics/mAP50(B)'].idxmax()]
    return round(float(b['metrics/mAP50(B)']),4), round(float(b['metrics/mAP50-95(B)']),4)
print(f"{'run':16s} {'mAP50':>8} {'mAP50-95':>9}")
for name, proj in [('base 40ep','runs/ctrl_base40'), ('base +10ep','runs/ctrl_base_plus10'),
                   ('YOLO11-FLEO','runs/y11'), ('YOLO26-FLEO','runs/y26')]:
    m = best(proj)
    print(f'{name:16s} {str(m[0]):>8} {str(m[1]):>9}')